 
 # Duplicate Question Detection using NLP and Deep Learning

## Problem Statement

Online Q&A platforms often receive duplicate questions that are phrased differently but have the same meaning. Detecting such duplicates helps avoid fragmented discussions and repeated answers.

The goal of this project is to build a model that predicts whether two questions have the same meaning.

## Approach

We use a hybrid approach:

1. Generate semantic embeddings using Sentence-BERT
2. Engineer features from these embeddings
3. Train a neural network classifier to predict duplicates

## Dataset

Quora Question Pairs Dataset

 

 


In [ ]:
# ======================================================
# Step 2: Import Required Libraries
# ======================================================

import pandas as pd
import numpy as np

# Sentence embedding model
from sentence_transformers import SentenceTransformer

# Dataset splitting
from sklearn.model_selection import train_test_split

# Evaluation metrics
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

# PyTorch for neural network
import torch
import torch.nn as nn
import torch.optim as optim

# Visualization
import matplotlib.pyplot as plt

print("All libraries imported successfully")

In [ ]:
# ======================================================
# Step 3: Load Dataset
# ======================================================

# Load dataset from CSV
df = pd.read_csv("quora_questions.csv")

# Select only required columns
df = df[['question1','question2','is_duplicate']]

# Remove missing rows
df = df.dropna()

# Use a subset for faster experimentation
df = df.sample(5000, random_state=42)

print("Dataset shape:", df.shape)

df.head()

In [ ]:
# ======================================================
# Step 4: Train/Test Split
# ======================================================

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['is_duplicate'],
    random_state=42
)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

In [ ]:
# ======================================================
# Step 5: Load Sentence Embedding Model
# ======================================================

model = SentenceTransformer('all-MiniLM-L6-v2')

print("Sentence-BERT model loaded successfully")

In [ ]:
# ======================================================
# Step 6: Generate Sentence Embeddings
# ======================================================

# Convert questions into vector representations

train_q1_emb = model.encode(train_df['question1'].tolist(), show_progress_bar=True)
train_q2_emb = model.encode(train_df['question2'].tolist(), show_progress_bar=True)

test_q1_emb = model.encode(test_df['question1'].tolist(), show_progress_bar=True)
test_q2_emb = model.encode(test_df['question2'].tolist(), show_progress_bar=True)

print("Embeddings generated successfully")

In [ ]:
 

# ======================================================
# Step 7: Feature Engineering + Normalization
# ======================================================

from sklearn.preprocessing import StandardScaler

# Combine embedding features
X_train = np.hstack([
    train_q1_emb,
    train_q2_emb,
    np.abs(train_q1_emb - train_q2_emb),
    train_q1_emb * train_q2_emb
])

X_test = np.hstack([
    test_q1_emb,
    test_q2_emb,
    np.abs(test_q1_emb - test_q2_emb),
    test_q1_emb * test_q2_emb
])

# Normalize features
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Labels
y_train = train_df['is_duplicate'].values
y_test = test_df['is_duplicate'].values

print("Feature shape:", X_train.shape)

In [ ]:
# ======================================================
# Step 8: Convert Data to PyTorch Tensors
# ======================================================

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1,1)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1,1)

In [ ]:
# ======================================================
# Step 9: Define Neural Network
# ======================================================

class DuplicateClassifier(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_dim,256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256,64),
            nn.ReLU(),

            nn.Linear(64,1),
            nn.Sigmoid()
        )

    def forward(self,x):

        return self.network(x)


input_dim = X_train.shape[1]

model_nn = DuplicateClassifier(input_dim)

print(model_nn)

In [ ]:
# ======================================================
# Step 10: Train Model
# ======================================================

criterion = nn.BCELoss()
optimizer = optim.Adam(model_nn.parameters(), lr=0.001)

epochs = 30   # Increased training time

for epoch in range(epochs):

    model_nn.train()

    optimizer.zero_grad()

    outputs = model_nn(X_train_tensor)

    loss = criterion(outputs, y_train_tensor)

    loss.backward()

    optimizer.step()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

In [ ]:
# ======================================================
# Step 11: Evaluate Model
# ======================================================

model_nn.eval()

with torch.no_grad():

    predictions = model_nn(X_test_tensor)

    predicted = (predictions > 0.5).float()

accuracy = accuracy_score(y_test, predicted.numpy())

print("Test Accuracy:", accuracy)

print("\nClassification Report:\n")

print(classification_report(y_test, predicted.numpy()))

In [ ]:
# ======================================================
# Step 12: Confusion Matrix
# ======================================================

ConfusionMatrixDisplay.from_predictions(
    y_test,
    predicted.numpy()
)

plt.title("Confusion Matrix")

plt.show()

In [ ]:
  
   # ======================================================
# Step 13: Test New Question Pairs
# ======================================================

def check_duplicate(q1, q2):

    # Generate embeddings
    emb1 = model.encode([q1])
    emb2 = model.encode([q2])

    # Create feature vector (same structure used in training)
    features = np.hstack([
        emb1,
        emb2,
        np.abs(emb1 - emb2),
        emb1 * emb2
    ])

    # Apply same scaler used during training
    features = scaler.transform(features)

    # Convert to tensor
    tensor = torch.tensor(features, dtype=torch.float32)

    # Predict
    with torch.no_grad():
        prediction = model_nn(tensor).item()

    print("Duplicate Probability:", prediction)

    if prediction > 0.4:
        print("Prediction: Duplicate Question")
    else:
        print("Prediction: Different Question")

In [ ]:
# ======================================================
# Step 14: Example Test
# ======================================================

check_duplicate(
    "How can I learn Python quickly?",
    "What is the fastest way to learn Python?"
)